In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Colab séparé — LR — Leak-Safe V3

Cette version corrige les fuites principales :
- split BGL par blocs disjoints **avant** la construction des fenêtres,
- Drain3 ajusté sur le train uniquement,
- OOF strict pour la phase fusion.


In [ ]:

# Colab — HDFS + BGL full + Dual Experts + Fusion (LR, Leak-Safe V3)

import sys, subprocess, pkgutil
def ensure_pkg(pkg, pip_name=None):
    pip_name = pip_name or pkg
    if pkgutil.find_loader(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for pkg, pip_name in [
    ("drain3", "drain3"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("tqdm", "tqdm"),
    ("xgboost", "xgboost"),
]:
    ensure_pkg(pkg, pip_name)

import os
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from scipy import sparse
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    brier_score_loss,
    roc_curve,
    precision_recall_curve,
)
from xgboost import XGBClassifier
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT_DIR = "/content/drive/MyDrive"
HDFS_LOG_NAME = "HDFS.log"
HDFS_LABEL_NAME = "anomaly_label.csv"
BGL_LOG_NAME = "BGL.log"

HDFS_SAMPLE_FRAC = 1.0
BGL_SAMPLE_FRAC = 1.0
BGL_WINDOW_SIZE = 40
BGL_STRIDE = 20
BGL_BLOCK_SIZE = 400  # blocked split to avoid train/test overlap leakage
TEST_SIZE = 0.20

TFIDF_MAX_FEATURES = 5000
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MIN_DF = 2
TEXT_SVD_COMPONENTS = 256
N_SPLITS_OOF = 5
MODEL_FAMILY = "lr"

print(f"Configuration chargée — modèle: {MODEL_FAMILY.upper()} — leak-safe V3")

def find_file(root_dir, filename):
    root = Path(root_dir)
    matches = list(root.rglob(filename))
    if not matches:
        return None
    matches = sorted(matches, key=lambda p: (len(str(p)), str(p)))
    return str(matches[0])

def print_distribution(df, name, label_col="label"):
    print(f"\n[{name}] shape={df.shape}")
    print(df[label_col].value_counts(dropna=False).sort_index())

def compute_scale_pos_weight(y):
    y = np.asarray(y).astype(int)
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    if n_pos == 0:
        return 1.0
    return max(1.0, n_neg / max(n_pos, 1))

def extract_block_id(text):
    m = re.search(r"(blk_-?\d+)", str(text))
    return m.group(1) if m else None

def normalize_text(text):
    text = str(text).strip()
    text = re.sub(r"0x[0-9a-fA-F]+", " HEX ", text)
    text = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", " IP ", text)
    text = re.sub(r"blk_-?\d+", " BLOCKID ", text)
    text = re.sub(r"\b\d{5,}\b", " NUM ", text)
    text = re.sub(r"\b\d+\b", " NUM ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower().strip()

def compute_metrics(y_true, y_proba, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba).astype(float)
    y_pred = (y_proba >= threshold).astype(int)
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        out["auc_roc"] = roc_auc_score(y_true, y_proba)
    except Exception:
        out["auc_roc"] = np.nan
    try:
        out["auc_pr"] = average_precision_score(y_true, y_proba)
    except Exception:
        out["auc_pr"] = np.nan
    try:
        out["brier"] = brier_score_loss(y_true, y_proba)
    except Exception:
        out["brier"] = np.nan
    return out

def build_template_miner():
    config = TemplateMinerConfig()
    return TemplateMiner(config=config)

def fit_parse_templates(train_df, text_col="messages"):
    miner = build_template_miner()
    templates_out = []
    for msgs in tqdm(train_df[text_col].tolist(), desc="Drain3 fit+parse train"):
        seq = []
        for msg in msgs:
            msg_norm = normalize_text(msg)
            result = miner.add_log_message(msg_norm)
            tpl = result.get("template_mined") if isinstance(result, dict) else None
            if tpl is None:
                tpl = msg_norm
            seq.append(tpl)
        templates_out.append(seq)
    out = train_df.copy()
    out["templates"] = templates_out
    out["sequence_text"] = [" [SEP] ".join(x) for x in templates_out]
    return out, miner

def transform_parse_templates(test_df, miner, text_col="messages"):
    templates_out = []
    for msgs in tqdm(test_df[text_col].tolist(), desc="Drain3 transform test"):
        seq = []
        for msg in msgs:
            msg_norm = normalize_text(msg)
            cluster = miner.match(msg_norm)
            if cluster is not None:
                tpl = cluster.get_template()
            else:
                tpl = msg_norm
            seq.append(tpl)
        templates_out.append(seq)
    out = test_df.copy()
    out["templates"] = templates_out
    out["sequence_text"] = [" [SEP] ".join(x) for x in templates_out]
    return out

def load_hdfs_labels(label_csv_path):
    df = pd.read_csv(label_csv_path)
    block_col, label_col = None, None
    for c in df.columns:
        cl = c.lower()
        if "block" in cl:
            block_col = c
        if "label" in cl:
            label_col = c
    if block_col is None:
        block_col = df.columns[0]
    if label_col is None:
        label_col = df.columns[1]
    out = df[[block_col, label_col]].copy()
    out.columns = ["block_id", "label_raw"]
    out["label"] = out["label_raw"].astype(str).str.lower().map(
        lambda x: 1 if ("anomaly" in x or "abnormal" in x or x == "1") else 0
    )
    out["block_id"] = out["block_id"].astype(str).str.strip()
    return out[["block_id", "label"]]

def build_hdfs_sessions(hdfs_log_path, hdfs_label_path):
    labels_df = load_hdfs_labels(hdfs_label_path)
    label_map = dict(zip(labels_df["block_id"], labels_df["label"]))
    groups = {}
    with open(hdfs_log_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, desc="Lecture HDFS"):
            line = line.rstrip("\n")
            block_id = extract_block_id(line)
            if block_id is None:
                continue
            if block_id not in label_map:
                continue
            groups.setdefault(block_id, []).append(line)
    rows = []
    for block_id, msgs in groups.items():
        rows.append({
            "sample_id": f"HDFS::{block_id}",
            "dataset": "HDFS",
            "group_id": block_id,
            "messages": msgs,
            "label": int(label_map[block_id]),
        })
    return pd.DataFrame(rows)

def safe_sample_hdfs(df, frac, label_col="label", seed=42):
    if frac >= 1.0:
        return df.copy().reset_index(drop=True)
    n_total = len(df)
    n_target = max(2, int(n_total * frac))
    y = df[label_col].values
    try:
        sampled_idx, _ = train_test_split(
            df.index, train_size=n_target, stratify=y, random_state=seed
        )
        return df.loc[sampled_idx].reset_index(drop=True)
    except Exception:
        return df.sample(n=min(n_target, n_total), random_state=seed).reset_index(drop=True)

def read_bgl_stream(bgl_log_path):
    lines, labels = [], []
    with open(bgl_log_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in tqdm(f, desc="Lecture BGL"):
            raw = raw.rstrip("\n")
            if not raw.strip():
                continue
            first = raw.split()[0]
            label = 0 if first == "-" else 1
            lines.append(raw)
            labels.append(label)
    return lines, labels

def make_bgl_blocks(lines, labels, block_size, min_len):
    blocks = []
    for block_id, start in enumerate(range(0, len(lines), block_size)):
        end = min(len(lines), start + block_size)
        block_lines = lines[start:end]
        block_labels = labels[start:end]
        if len(block_lines) < min_len:
            continue
        blocks.append({
            "block_id": block_id,
            "start": start,
            "end": end,
            "lines": block_lines,
            "labels": block_labels,
            "label": int(max(block_labels)),
        })
    return blocks

def sample_bgl_blocks(blocks, frac, seed=42):
    if frac >= 1.0:
        return list(blocks)
    idx = np.arange(len(blocks))
    y = np.array([b["label"] for b in blocks], dtype=int)
    n_target = max(2, int(len(blocks) * frac))
    try:
        sampled_idx, _ = train_test_split(idx, train_size=n_target, stratify=y, random_state=seed)
        sampled_idx = set(sampled_idx.tolist())
        return [b for i, b in enumerate(blocks) if i in sampled_idx]
    except Exception:
        random.Random(seed).shuffle(blocks)
        return blocks[:n_target]

def split_bgl_blocks(blocks, test_size=0.2, seed=42):
    idx = np.arange(len(blocks))
    y = np.array([b["label"] for b in blocks], dtype=int)
    try:
        tr_idx, te_idx = train_test_split(idx, test_size=test_size, stratify=y, random_state=seed)
    except Exception:
        tr_idx, te_idx = train_test_split(idx, test_size=test_size, random_state=seed)
    tr_set = set(tr_idx.tolist())
    te_set = set(te_idx.tolist())
    train_blocks = [b for i, b in enumerate(blocks) if i in tr_set]
    test_blocks = [b for i, b in enumerate(blocks) if i in te_set]
    return train_blocks, test_blocks

def build_windows_from_blocks(blocks, window_size=40, stride=20):
    rows = []
    for block in tqdm(blocks, desc="Fenêtres BGL par blocs"):
        lines = block["lines"]
        labels = block["labels"]
        for local_start in range(0, max(0, len(lines) - window_size + 1), stride):
            local_end = local_start + window_size
            win_lines = lines[local_start:local_end]
            win_labels = labels[local_start:local_end]
            global_start = block["start"] + local_start
            global_end = block["start"] + local_end
            rows.append({
                "sample_id": f"BGL::{block['block_id']}_{global_start}_{global_end}",
                "dataset": "BGL",
                "group_id": f"{block['block_id']}:{global_start}_{global_end}",
                "messages": win_lines,
                "label": int(max(win_labels)),
            })
    return pd.DataFrame(rows)

ANOMALY_TERMS = [
    "error", "exception", "fatal", "fail", "failed", "warn", "warning",
    "abort", "panic", "timeout", "timed out", "unable", "refused"
]
ANOMALY_PATTERN = re.compile("|".join(re.escape(x) for x in ANOMALY_TERMS), re.IGNORECASE)

def add_sequence_features(df):
    df = df.copy()
    df["seq_len_raw"] = df["messages"].apply(len)
    df["n_templates"] = df["templates"].apply(len)
    df["n_unique_templates"] = df["templates"].apply(lambda x: len(set(x)))
    df["anomaly_keyword_count"] = df["sequence_text"].apply(lambda x: len(ANOMALY_PATTERN.findall(x)))
    df["anomaly_keyword_ratio"] = df["anomaly_keyword_count"] / df["n_templates"].clip(lower=1)
    df["avg_template_len"] = df["templates"].apply(
        lambda seq: float(np.mean([len(t.split()) for t in seq])) if len(seq) else 0.0
    )
    return df

STAT_COLS = [
    "seq_len_raw",
    "n_templates",
    "n_unique_templates",
    "anomaly_keyword_count",
    "anomaly_keyword_ratio",
    "avg_template_len",
]

class FeatureBuilder:
    def __init__(self, model_family="lr"):
        self.model_family = model_family
        self.vectorizer = TfidfVectorizer(
            max_features=TFIDF_MAX_FEATURES,
            ngram_range=TFIDF_NGRAM_RANGE,
            min_df=TFIDF_MIN_DF,
            lowercase=False,
        )
        self.scaler = StandardScaler()
        self.svd = None

    def fit_transform(self, df):
        X_text = self.vectorizer.fit_transform(df["sequence_text"].tolist())
        X_stats = self.scaler.fit_transform(df[STAT_COLS].values)
        if self.model_family in {"rf", "xgb"}:
            n_comp = min(TEXT_SVD_COMPONENTS, max(2, X_text.shape[1] - 1))
            self.svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
            X_text = self.svd.fit_transform(X_text)
            return np.hstack([X_text, X_stats])
        return sparse.hstack([X_text, sparse.csr_matrix(X_stats)], format="csr")

    def transform(self, df):
        X_text = self.vectorizer.transform(df["sequence_text"].tolist())
        X_stats = self.scaler.transform(df[STAT_COLS].values)
        if self.model_family in {"rf", "xgb"}:
            X_text = self.svd.transform(X_text)
            return np.hstack([X_text, X_stats])
        return sparse.hstack([X_text, sparse.csr_matrix(X_stats)], format="csr")

def make_base_model(family, y_train=None):
    family = family.lower()
    if family == "lr":
        return LogisticRegression(max_iter=3000, solver="liblinear", random_state=SEED, class_weight="balanced")
    if family == "svm":
        base = LinearSVC(C=1.0, random_state=SEED, class_weight="balanced")
        return CalibratedClassifierCV(base, method="sigmoid", cv=3)
    if family == "rf":
        return RandomForestClassifier(
            n_estimators=350, max_depth=None, min_samples_split=2,
            min_samples_leaf=1, n_jobs=-1, random_state=SEED,
            class_weight="balanced_subsample",
        )
    if family == "xgb":
        spw = compute_scale_pos_weight(y_train) if y_train is not None else 1.0
        return XGBClassifier(
            n_estimators=350, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            objective="binary:logistic", eval_metric="logloss",
            random_state=SEED, n_jobs=-1, scale_pos_weight=spw,
        )
    raise ValueError(f"Famille inconnue: {family}")

def predict_proba_1(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        scores = np.asarray(scores, dtype=float)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
        return scores
    pred = model.predict(X)
    return np.asarray(pred, dtype=float)

def make_fusion_model(family, y_train=None):
    return make_base_model(family, y_train)

def build_fusion_matrix(df, p_hdfs, p_bgl, include_domain_flag=True):
    X = df[STAT_COLS].copy().reset_index(drop=True)
    X.insert(0, "p_hdfs", np.asarray(p_hdfs, dtype=float))
    X.insert(1, "p_bgl", np.asarray(p_bgl, dtype=float))
    if include_domain_flag:
        X["domain_flag"] = (df["dataset"].values == "BGL").astype(int)
    return X

def get_strict_oof_predictions(train_df, family, n_splits=5):
    y = train_df["label"].values.astype(int)
    oof = np.zeros(len(train_df), dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    idx_all = np.arange(len(train_df))
    for tr_idx, va_idx in skf.split(idx_all, y):
        tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
        va_df = train_df.iloc[va_idx].reset_index(drop=True)
        fb = FeatureBuilder(family)
        X_tr = fb.fit_transform(tr_df)
        X_va = fb.transform(va_df)
        model = make_base_model(family, tr_df["label"].values.astype(int))
        model.fit(X_tr, tr_df["label"].values.astype(int))
        oof[va_idx] = predict_proba_1(model, X_va)
    return oof

def plot_curves_and_cm(y_true, y_proba, title_prefix="Model"):
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba).astype(float)
    y_pred = (y_proba >= 0.5).astype(int)

    try:
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        plt.figure(figsize=(6, 4))
        plt.plot(fpr, tpr, label=f"ROC AUC={roc_auc_score(y_true, y_proba):.4f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC — {title_prefix}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
    except Exception:
        pass

    try:
        prec, rec, _ = precision_recall_curve(y_true, y_proba)
        plt.figure(figsize=(6, 4))
        plt.plot(rec, prec, label=f"PR AUC={average_precision_score(y_true, y_proba):.4f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"PR Curve — {title_prefix}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
    except Exception:
        pass

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Confusion Matrix — {title_prefix}")
    plt.colorbar()
    ticks = np.arange(2)
    plt.xticks(ticks, ["Normal", "Anomaly"])
    plt.yticks(ticks, ["Normal", "Anomaly"])
    thresh = cm.max() / 2.0 if cm.max() else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()

def evaluate_pipeline_family(family, hdfs_train, hdfs_test, bgl_train, bgl_test):
    print(f"\n================ {family.upper()} ================")

    fb_hdfs = FeatureBuilder(family)
    Xh_tr = fb_hdfs.fit_transform(hdfs_train)
    Xh_te = fb_hdfs.transform(hdfs_test)
    yh_tr = hdfs_train["label"].values.astype(int)
    yh_te = hdfs_test["label"].values.astype(int)

    fb_bgl = FeatureBuilder(family)
    Xb_tr = fb_bgl.fit_transform(bgl_train)
    Xb_te = fb_bgl.transform(bgl_test)
    yb_tr = bgl_train["label"].values.astype(int)
    yb_te = bgl_test["label"].values.astype(int)

    model_hdfs = make_base_model(family, yh_tr)
    model_bgl = make_base_model(family, yb_tr)
    model_hdfs.fit(Xh_tr, yh_tr)
    model_bgl.fit(Xb_tr, yb_tr)

    ph_hdfs_on_hdfs = predict_proba_1(model_hdfs, Xh_te)
    pb_bgl_on_bgl = predict_proba_1(model_bgl, Xb_te)
    ph_hdfs_on_bgl = predict_proba_1(model_hdfs, fb_hdfs.transform(bgl_test))
    pb_bgl_on_hdfs = predict_proba_1(model_bgl, fb_bgl.transform(hdfs_test))

    oof_hdfs = get_strict_oof_predictions(hdfs_train, family, n_splits=N_SPLITS_OOF)
    oof_bgl = get_strict_oof_predictions(bgl_train, family, n_splits=N_SPLITS_OOF)

    # cross-domain train scores are generated by experts trained on the opposite-domain train only
    p_bgl_on_hdfs_train = predict_proba_1(model_bgl, fb_bgl.transform(hdfs_train))
    p_hdfs_on_bgl_train = predict_proba_1(model_hdfs, fb_hdfs.transform(bgl_train))

    fusion_train_hdfs = build_fusion_matrix(hdfs_train, oof_hdfs, p_bgl_on_hdfs_train, include_domain_flag=True)
    fusion_train_bgl = build_fusion_matrix(bgl_train, p_hdfs_on_bgl_train, oof_bgl, include_domain_flag=True)
    fusion_train = pd.concat([fusion_train_hdfs, fusion_train_bgl], axis=0).reset_index(drop=True)
    y_fusion_train = np.concatenate([yh_tr, yb_tr])

    fusion_test_hdfs = build_fusion_matrix(hdfs_test, ph_hdfs_on_hdfs, pb_bgl_on_hdfs, include_domain_flag=True)
    fusion_test_bgl = build_fusion_matrix(bgl_test, ph_hdfs_on_bgl, pb_bgl_on_bgl, include_domain_flag=True)
    fusion_test_all = pd.concat([fusion_test_hdfs, fusion_test_bgl], axis=0).reset_index(drop=True)
    y_fusion_test = np.concatenate([yh_te, yb_te])

    scaler_fusion = StandardScaler()
    X_fusion_train = scaler_fusion.fit_transform(fusion_train.values)
    X_fusion_test_hdfs = scaler_fusion.transform(fusion_test_hdfs.values)
    X_fusion_test_bgl = scaler_fusion.transform(fusion_test_bgl.values)
    X_fusion_test_all = scaler_fusion.transform(fusion_test_all.values)

    fusion_model = make_fusion_model(family, y_fusion_train)
    fusion_model.fit(X_fusion_train, y_fusion_train)

    pf_hdfs = predict_proba_1(fusion_model, X_fusion_test_hdfs)
    pf_bgl = predict_proba_1(fusion_model, X_fusion_test_bgl)
    pf_all = predict_proba_1(fusion_model, X_fusion_test_all)

    rows = []
    evals = [
        (f"{family.upper()}_HDFS_expert_on_HDFS", yh_te, ph_hdfs_on_hdfs),
        (f"{family.upper()}_BGL_expert_on_BGL", yb_te, pb_bgl_on_bgl),
        (f"{family.upper()}_HDFS_expert_on_BGL", yb_te, ph_hdfs_on_bgl),
        (f"{family.upper()}_BGL_expert_on_HDFS", yh_te, pb_bgl_on_hdfs),
        (f"{family.upper()}_Fusion_on_HDFS", yh_te, pf_hdfs),
        (f"{family.upper()}_Fusion_on_BGL", yb_te, pf_bgl),
        (f"{family.upper()}_Fusion_on_Combined", y_fusion_test, pf_all),
    ]
    curve_payload = {}
    for name, y_true, y_proba in evals:
        row = {"model": name}
        row.update(compute_metrics(y_true, y_proba))
        rows.append(row)
        curve_payload[name] = (y_true, y_proba)

    return pd.DataFrame(rows), curve_payload

# ----------------------------
# Data loading and leak-safe splitting
# ----------------------------
hdfs_log_path = find_file(ROOT_DIR, HDFS_LOG_NAME)
hdfs_label_path = find_file(ROOT_DIR, HDFS_LABEL_NAME)
bgl_log_path = find_file(ROOT_DIR, BGL_LOG_NAME)

print("HDFS log    :", hdfs_log_path)
print("HDFS labels :", hdfs_label_path)
print("BGL log     :", bgl_log_path)
assert hdfs_log_path is not None, f"{HDFS_LOG_NAME} introuvable sous {ROOT_DIR}"
assert hdfs_label_path is not None, f"{HDFS_LABEL_NAME} introuvable sous {ROOT_DIR}"
assert bgl_log_path is not None, f"{BGL_LOG_NAME} introuvable sous {ROOT_DIR}"

# HDFS: build grouped sessions first, then split (safe at block/session level)
hdfs_df = build_hdfs_sessions(hdfs_log_path, hdfs_label_path)
print_distribution(hdfs_df, "HDFS brut")
hdfs_df = safe_sample_hdfs(hdfs_df, HDFS_SAMPLE_FRAC, label_col="label", seed=SEED)
print_distribution(hdfs_df, "HDFS sample")

hdfs_train_raw, hdfs_test_raw = train_test_split(
    hdfs_df, test_size=TEST_SIZE, random_state=SEED, stratify=hdfs_df["label"]
)
hdfs_train_raw = hdfs_train_raw.reset_index(drop=True)
hdfs_test_raw = hdfs_test_raw.reset_index(drop=True)

# BGL: blocked split on disjoint raw-line chunks BEFORE window construction to avoid overlap leakage
bgl_lines, bgl_labels = read_bgl_stream(bgl_log_path)
bgl_blocks = make_bgl_blocks(bgl_lines, bgl_labels, block_size=BGL_BLOCK_SIZE, min_len=BGL_WINDOW_SIZE)
bgl_blocks = sample_bgl_blocks(bgl_blocks, frac=BGL_SAMPLE_FRAC, seed=SEED)
bgl_train_blocks, bgl_test_blocks = split_bgl_blocks(bgl_blocks, test_size=TEST_SIZE, seed=SEED)
bgl_train_raw = build_windows_from_blocks(bgl_train_blocks, window_size=BGL_WINDOW_SIZE, stride=BGL_STRIDE).reset_index(drop=True)
bgl_test_raw = build_windows_from_blocks(bgl_test_blocks, window_size=BGL_WINDOW_SIZE, stride=BGL_STRIDE).reset_index(drop=True)

print_distribution(hdfs_train_raw, "HDFS train raw")
print_distribution(hdfs_test_raw, "HDFS test raw")
print_distribution(bgl_train_raw, "BGL train raw")
print_distribution(bgl_test_raw, "BGL test raw")
print("\nLeak protections actives: split HDFS au niveau session, split BGL par blocs disjoints avant fenêtres, Drain3 fit sur train seulement, OOF strict pour la fusion.")

# ----------------------------
# Train-only parsing / test-only transform
# ----------------------------
hdfs_train, hdfs_miner = fit_parse_templates(hdfs_train_raw, text_col="messages")
hdfs_test = transform_parse_templates(hdfs_test_raw, hdfs_miner, text_col="messages")
bgl_train, bgl_miner = fit_parse_templates(bgl_train_raw, text_col="messages")
bgl_test = transform_parse_templates(bgl_test_raw, bgl_miner, text_col="messages")

hdfs_train = add_sequence_features(hdfs_train)
hdfs_test = add_sequence_features(hdfs_test)
bgl_train = add_sequence_features(bgl_train)
bgl_test = add_sequence_features(bgl_test)

print_distribution(hdfs_train, "HDFS train final")
print_distribution(hdfs_test, "HDFS test final")
print_distribution(bgl_train, "BGL train final")
print_distribution(bgl_test, "BGL test final")
print("\nAucun resampling n'est appliqué. La gestion du déséquilibre se fait par class weights.")

results_df, curves = evaluate_pipeline_family(MODEL_FAMILY, hdfs_train, hdfs_test, bgl_train, bgl_test)
pd.set_option("display.max_columns", None)
display(results_df.sort_values("f1", ascending=False).reset_index(drop=True))

best_name = results_df.sort_values("f1", ascending=False)["model"].iloc[0]
print(f"\nCourbes pour le meilleur bloc: {best_name}")
y_true_best, y_proba_best = curves[best_name]
plot_curves_and_cm(y_true_best, y_proba_best, title_prefix=best_name)
